# Importamos librerias:

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

# Importamos la base de datos:

In [ ]:
#Importamos la base de datos
df = pd.read_csv('boston.csv')

#Eliminamos la columna 'Unnamed: 0'
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns=['Unnamed: 0'])

#Obtenemos las medidas descriptivas
print("Medidas Descriptivas:")
print(df.describe())
print("\n")


Medidas Descriptivas:
             crim          zn       indus        chas         nox          rm  \
count  506.000000  506.000000  506.000000  506.000000  506.000000  506.000000   
mean     3.613524   11.363636   11.136779    0.069170    0.554695    6.284634   
std      8.601545   23.322453    6.860353    0.253994    0.115878    0.702617   
min      0.006320    0.000000    0.460000    0.000000    0.385000    3.561000   
25%      0.082045    0.000000    5.190000    0.000000    0.449000    5.885500   
50%      0.256510    0.000000    9.690000    0.000000    0.538000    6.208500   
75%      3.677083   12.500000   18.100000    0.000000    0.624000    6.623500   
max     88.976200  100.000000   27.740000    1.000000    0.871000    8.780000   

              age         dis         rad         tax     ptratio       black  \
count  506.000000  506.000000  506.000000  506.000000  506.000000  506.000000   
mean    68.574901    3.795043    9.549407  408.237154   18.455534  356.674032   
std  

# División de los datos para entrenamiento y testing

In [17]:
#División de la muestra
#Separamos los atributos (X) del vector objetivo (y)
X = df.drop(columns=['medv'])
y = df['medv']
#Generamos conjuntos de entrenamiento y validación (33% validación, semilla 42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)


# Generación de los modelos

In [ ]:
#Ejercicio 3: Generación de modelos
#Inicializamos ambos modelos
modelo_con_intercepto = LinearRegression(fit_intercept=True)
modelo_sin_intercepto = LinearRegression(fit_intercept=False)

#Entrenamos los modelos
modelo_con_intercepto.fit(X_train, y_train)
modelo_sin_intercepto.fit(X_train, y_train)

#Generamos las predicciones con el set de validación
y_pred_con = modelo_con_intercepto.predict(X_test)
y_pred_sin = modelo_sin_intercepto.predict(X_test)

# Obtención de las métricas:

In [ ]:
#Ejercicio 4: Obtención de métricas
def report_scores(y_pred, y_test):
    """
    Imprime las métricas de Error Cuadrático Promedio (MSE) y R2.
    """
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    print(f"MSE: {mse:.4f}")
    print(f"R2 : {r2:.4f}\n")
    return mse, r2

print("Métricas Modelo con Intercepto:")
mse_con, r2_con = report_scores(y_pred_con, y_test)

print("Métricas Modelo sin Intercepto")
mse_sin, r2_sin = report_scores(y_pred_sin, y_test)

Métricas Modelo con Intercepto:
MSE: 20.7240
R2 : 0.7262

Métricas Modelo sin Intercepto
MSE: 23.6038
R2 : 0.6881



**Nota: El mejor modelo el modelo CON intercepto, ya que presenta un menor MSE y un mayor $R^2$**

# Refactorización del modelo:

In [11]:
#Ejercicio 5: Refactorización del modelo
def fetch_features(dataframe, target_col='medv'):
    """
    Retorna una lista con las correlaciones absolutas de cada atributo 
    con el vector objetivo, ordenadas de mayor a menor.
    """
    correlations = []
    for col in dataframe.columns:
        if col != target_col:
            # Correlación de Pearson
            corr = dataframe[col].corr(dataframe[target_col])
            correlations.append((corr, col))
            
    # Ordenamos por valor absoluto de mayor a menor
    correlations_sorted = sorted(correlations, key=lambda x: abs(x[0]), reverse=True)
    return correlations_sorted

correlaciones = fetch_features(df, 'medv')

print("'Top 6 atributos con mayor correlación con 'medv'")
for corr, col in correlaciones[:6]:
    print(f"Atributo: {col}, Correlación: {corr:.4f}")
print("\n")

#Extraemos los nombres de estos 6 atributos.
#Notar que extraemos exactamente los identificados: lstat, rm, ptratio, indus, tax, nox.
top_6_features = [col for corr, col in correlaciones[:6]]

'Top 6 atributos con mayor correlación con 'medv'
Atributo: lstat, Correlación: -0.7377
Atributo: rm, Correlación: 0.6954
Atributo: ptratio, Correlación: -0.5078
Atributo: indus, Correlación: -0.4837
Atributo: tax, Correlación: -0.4685
Atributo: nox, Correlación: -0.4273




In [20]:
#Refactorización del modelo predictivo
#Generamos la nueva matriz X usando sólo los 6 mejores atributos.
ordered_top_6 = ['lstat', 'ptratio', 'rm', 'indus', 'tax', 'nox']
X_top = df[ordered_top_6]

X_train_top, X_test_top, y_train_top, y_test_top = train_test_split(X_top, y, test_size=0.33, random_state=42)

#Entrenamos un nuevo modelo con intercepto
mejor_modelo_refactorizado = LinearRegression(fit_intercept=True)
mejor_modelo_refactorizado.fit(X_train_top, y_train_top)
y_pred_top = mejor_modelo_refactorizado.predict(X_test_top)

print("Métricas: Nuevo Modelo Refactorizado (Top 6 Atributos)")
report_scores(y_pred_top, y_test_top)

Métricas: Nuevo Modelo Refactorizado (Top 6 Atributos)
MSE: 24.9996
R2 : 0.6697



(24.999580766427155, 0.6696607622948445)

# Predicción:

In [19]:
#Predicción de casos
worst_neighbor = np.array([ 37.9 , 12.6 , 3.5 , 27.7 , 187 , 0.87 ]).reshape( 1 ,-1 )
best_neighbor = np.array([ 1.73 , 22 , 8.7 , 0.46 , 711 , 0.38 ]).reshape( 1 ,-1 )

#Convertimos los arreglos a DataFrames
worst_df = pd.DataFrame(worst_neighbor, columns=ordered_top_6)
best_df = pd.DataFrame(best_neighbor, columns=ordered_top_6)

pred_worst = mejor_modelo_refactorizado.predict(worst_df)[0]
pred_best = mejor_modelo_refactorizado.predict(best_df)[0]

print("Predicciones de Escenarios")
print(f"Valor esperado en el PEOR vecindario  : ${pred_worst * 1000:,.2f} USD")
print(f"Valor esperado en el MEJOR vecindario : ${pred_best * 1000:,.2f} USD")

Predicciones de Escenarios
Valor esperado en el PEOR vecindario  : $1,293.03 USD
Valor esperado en el MEJOR vecindario : $35,415.14 USD
